In [1]:
import requests
import pandas as pd

# Data Collection

**Dataset: Kalshi KXNASDAQ100 Historical Markets**

This dataset contains resolved Kalshi prediction markets from a single day (March 12, 2025), each asking whether the Nasdaq-100 closed within a specific price range, along with their trading data and binary yes/no outcomes.

NOTE: This data isn't nearly enough for training a model yet, so we will have to work on gathering more data.

In [3]:
all_markets: list[dict] = []
cursor = ""

while True:
    # build request url with cursor for pagination
    url = f"https://api.elections.kalshi.com/trade-api/v2/historical/markets?series_ticker=KXNASDAQ100&limit=100&cursor={cursor}"
    response = requests.get(url)
    data: dict = response.json()

    # stop if no markets returned
    markets: list[dict] = data.get('markets', [])
    if not markets:
        break

    # add current page of markers to full list
    all_markets.extend(markets)

    # move to next page or stop if no cursor is returned
    cursor = data.get('cursor', '')
    if not cursor:
        break

print(f"Total markets: {len(all_markets)}")

Total markets: 500


# Data Extraction
Extract relevant fields from each market dictionary:

In [4]:
# each row represents one yes/no kalshi market with its outcome and trading data
rows: list[dict] = []

for market in all_markets:
    rows.append({
        'ticker': market.get('ticker'),
        'title': market.get('title'),  # yes/no question
        'result': market.get('result'),  # yes/no outcome
        'yes_bid': market.get('yes_bid_dollars'),  # highest buy price for yes
        'yes_ask': market.get('yes_ask_dollars'),  # lowest sell price for yes
        'volume': market.get('volume_fp'),  # total contracts traded
        'open_time': market.get('open_time'),
        'close_time': market.get('close_time'),
        'expiration_time': market.get('expiration_time'),  # when market expired
        'last_price': market.get('last_price_dollars'),  # final traded price
    })

df = pd.DataFrame(rows)

df['close_time'] = pd.to_datetime(df['close_time'], format='ISO8601')

In [5]:
print(f"DataFrame shape: {df.shape}")
print(df.head())

DataFrame shape: (500, 10)
                  ticker                                              title  \
0    KXATPIWO-25CERDE-DE  Will Alex de Minaur be a winner of the round o...   
1   KXATPIWO-25CERDE-CER  Will Francisco Cerundolo be a winner of the ro...   
2  KXATPIWO-25NAKSHE-SHE  Will Ben Shelton be a winner of the round of 1...   
3  KXATPIWO-25NAKSHE-NAK  Will Brandon Nakashima be a winner of the roun...   
4  KXWTAIWO-25GAUBEN-BEN  Will Belinda Bencic be a winner of the round o...   

  result yes_bid yes_ask   volume             open_time  \
0     no  0.0000  0.9900   227.00  2025-03-12T18:00:00Z   
1    yes  0.9500  1.0000   820.00  2025-03-12T18:00:00Z   
2    yes  0.9900  1.0000   120.00  2025-03-12T18:00:00Z   
3     no  0.0000  0.5000   618.00  2025-03-12T18:00:00Z   
4    yes  0.1600  1.0000  6479.00  2025-03-12T18:00:00Z   

                        close_time       expiration_time last_price  
0 2025-03-12 22:12:20.782005+00:00  2027-03-12T19:10:00Z     0.0100  
1 2

# Data Cleaning

In [6]:
# convert price and volume columns from str to float
df['yes_bid'] = df['yes_bid'].astype(float)
df['yes_ask'] = df['yes_ask'].astype(float)
df['volume'] = df['volume'].astype(float)
df['last_price'] = df['last_price'].astype(float)

In [7]:
# convert time columns to datetime
df['open_time'] = pd.to_datetime(df['open_time'], format='ISO8601')
df['close_time'] = pd.to_datetime(df['close_time'], format='ISO8601')
df['expiration_time'] = pd.to_datetime(df['expiration_time'], format='ISO8601')

In [8]:
# compute time to expiration in hours (float)
df['time_to_expiration'] = (df['expiration_time'] - df['open_time']).dt.total_seconds() / 3600

In [9]:
# encode target variable: yes -> 1, no -> 0
df['result'] = df['result'].map({'yes': 1, 'no': 0})

In [10]:
# check for missing values
print(df.isnull().sum())

ticker                0
title                 0
result                0
yes_bid               0
yes_ask               0
volume                0
open_time             0
close_time            0
expiration_time       0
last_price            0
time_to_expiration    0
dtype: int64
